## Inspection of London Airbnb dataset

Target: Getting to know the details of the dataset before selecting and moving to the cleaning methods

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 30)
print("Libraries imported successfully")

Libraries imported successfully


In [24]:
file_path = Path("../data/raw/listings.csv.gz")

print("Checking:", file_path.resolve())
print("Dataset found:", file_path.exists())

Checking: C:\Users\franc\Downloads\AIML assessment group\data\raw\listings.csv.gz
Dataset found: True


In [26]:
df = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

Dataset loaded successfully
Dataset shape: (91954, 90)


C:\Users\franc\AppData\Local\Temp\ipykernel_69568\3772534578.py:1: DtypeWarning: Columns (4: source) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [37]:
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

df.info()

print("\nCollection dates:")
print(df["last_scraped"].value_counts(dropna=False).sort_index())

Rows and columns: (91954, 90)

Column names:
['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_pric

In [38]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

display(missing_summary.round(2))

print("\nCompletely empty columns:")
print(df.columns[df.isna().all()].tolist())

,missing_count,missing_percent
neighborhood_overview,91954,100.0
host_since,91954,100.0
host_response_time,91954,100.0
host_thumbnail_url,91954,100.0
host_acceptance_rate,91954,100.0
...,...,...
availability_30,0,0.0
calculated_host_listings_count,0,0.0
calculated_host_listings_count_entire_homes,0,0.0
calculated_host_listings_count_private_rooms,0,0.0



Completely empty columns:
['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'bathrooms', 'calendar_updated', 'license', 'instant_bookable']


In [39]:
print("Missing listing IDs:", df["id"].isna().sum())
print("Repeated listing IDs:", df["id"].duplicated().sum())
print("Exact duplicate rows:", df.duplicated().sum())

Missing listing IDs: 0
Repeated listing IDs: 0
Exact duplicate rows: 0


In [40]:
print("Stored price type:", df["price"].dtype)
display(df["price"].head(10))


Stored price type: str


0    $267.50
1    $127.00
2    $137.50
3    $228.00
4        NaN
5    $290.00
6    $517.50
7        NaN
8        NaN
9     $59.00
Name: price, dtype: str

In [41]:
price_check = pd.to_numeric(
    df["price"].astype("string")
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)

parse_failures = df["price"].notna() & price_check.isna()

print("Non-empty values that failed conversion:", parse_failures.sum())
print("Missing prices:", price_check.isna().sum())
print("Zero or negative prices:", (price_check <= 0).sum())

display(
    price_check.describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

Non-empty values that failed conversion: 0
Missing prices: 32049
Zero or negative prices: 0


count       59905.0
mean     248.354667
std      577.024272
min            10.0
1%             37.0
25%           99.67
50%           175.0
75%          279.75
95%         604.986
99%         1272.16
max         75000.0
Name: price, dtype: Float64

In [42]:
highest_price_indices = price_check.nlargest(10).index

display(
    df.loc[highest_price_indices, [
        "price",
        "room_type",
        "property_type",
        "accommodates",
        "bedrooms",
        "bathrooms_text",
        "neighbourhood_cleansed"
    ]]
)

,price,room_type,property_type,accommodates,bedrooms,bathrooms_text,neighbourhood_cleansed
9263,"$75,000.00",Private room,Private room in rental unit,1,1.0,1 shared bath,Kensington and Chelsea
59223,"$59,214.00",Private room,Private room in rental unit,3,1.0,1 shared bath,Lambeth
53774,"$28,000.00",Private room,Private room in rental unit,3,1.0,1 bath,Lambeth
53920,"$23,686.00",Private room,Private room in rental unit,3,1.0,1 bath,Lambeth
54210,"$23,520.00",Entire home/apt,Entire villa,16,22.0,20 baths,Westminster
61863,"$16,318.40",Entire home/apt,Entire rental unit,2,1.0,1 bath,Westminster
55464,"$14,744.00",Entire home/apt,Entire home,16,6.0,4.5 baths,Islington
42564,"$12,000.00",Entire home/apt,Entire rental unit,2,1.0,1 bath,Wandsworth
60961,"$10,708.00",Entire home/apt,Entire rental unit,16,24.0,24 baths,Camden
34324,"$10,679.00",Entire home/apt,Entire serviced apartment,10,5.0,5 baths,Westminster


In [43]:
price_missing_check = pd.DataFrame({
    "room_type": df["room_type"],
    "source": df["source"].fillna("(missing source)"),
    "price_missing": price_check.isna()
})

for column in ["room_type", "source"]:
    summary = (
        price_missing_check.groupby(column)["price_missing"]
        .agg(["size", "sum", "mean"])
        .rename(columns={
            "size": "total_listings",
            "sum": "missing_prices",
            "mean": "missing_percent"
        })
    )

    summary["missing_percent"] *= 100
    display(summary.round(2))

,total_listings,missing_prices,missing_percent
room_type,,,
Entire home/apt,61074,20553,33.65
Hotel room,207,73,35.27
Private room,30471,11386,37.37
Shared room,202,37,18.32


,total_listings,missing_prices,missing_percent
source,,,
(missing source),59643,2325,3.90
previous scrape,32311,29724,91.99


In [44]:
numeric_columns = [
    "accommodates",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "availability_365"
]

display(df[numeric_columns].describe().T)

print("\nZero values:")
display(df[numeric_columns].eq(0).sum())

print("\nNegative values:")
display(df[numeric_columns].lt(0).sum())

print("\nBathroom descriptions:")
display(df["bathrooms_text"].value_counts(dropna=False))

print(
    "\nNumeric bathroom value missing but description available:",
    (df["bathrooms"].isna() & df["bathrooms_text"].notna()).sum()
)

,count,mean,std,min,25%,50%,75%,max
accommodates,91954.0,3.405801,2.132319e+00,1.0,2.0,3.0,4.0,1.600000e+01
bedrooms,91909.0,1.540546,9.998376e-01,0.0,1.0,1.0,2.0,3.000000e+01
beds,89964.0,1.936552,1.360442e+00,0.0,1.0,1.0,2.0,5.000000e+01
minimum_nights,91645.0,4.717573,2.130360e+01,1.0,1.0,2.0,3.0,1.125000e+03
maximum_nights,91645.0,334310.065503,2.659697e+07,1.0,99.0,365.0,1125.0,2.147484e+09
availability_365,91954.0,153.185408,1.436767e+02,0.0,0.0,120.0,305.0,3.650000e+02



Zero values:


accommodates            0
bedrooms             3264
beds                    1
minimum_nights          0
maximum_nights          0
availability_365    28282
dtype: int64


Negative values:


accommodates        0
bedrooms            0
beds                0
minimum_nights      0
maximum_nights      0
availability_365    0
dtype: int64


Bathroom descriptions:


bathrooms_text
1 bath               42102
2 baths              12601
1 shared bath        11102
1 private bath        8627
1.5 baths             5135
2.5 baths             2705
1.5 shared baths      2468
3 baths               1736
2 shared baths        1446
3.5 baths              737
Shared half-bath       460
4 baths                374
0 baths                364
2.5 shared baths       320
0 shared baths         307
Half-bath              273
Private half-bath      254
3 shared baths         243
4.5 baths              186
5 baths                130
NaN                    108
5.5 baths               46
6 baths                 40
3.5 shared baths        33
4 shared baths          25
6 shared baths          19
6.5 baths               16
4.5 shared baths        14
7 baths                 12
5 shared baths          11
No bathroom             10
8 baths                  8
5.5 shared baths         8
10 shared baths          7
10 baths                 6
9 baths                  4
12 baths     


Numeric bathroom value missing but description available: 91846


In [45]:
for column in [
    "room_type",
    "property_type",
    "neighbourhood_cleansed"
]:
    print(f"\n{column}:")
    display(df[column].value_counts(dropna=False))

display(
    pd.crosstab(
        df["number_of_reviews"].eq(0),
        df["review_scores_rating"].isna(),
        rownames=["Has zero reviews"],
        colnames=["Rating is missing"],
        dropna=False
    )
)


room_type:


room_type
Entire home/apt    61074
Private room       30471
Hotel room           207
Shared room          202
Name: count, dtype: int64


property_type:


property_type
Entire rental unit             40076
Private room in rental unit    12685
Private room in home           10602
Entire home                     9244
Entire condo                    7732
                               ...  
Lighthouse                         1
Shipping container                 1
Shared room in farm stay           1
Private room in camper/rv          1
Cycladic home                      1
Name: count, Length: 89, dtype: int64


neighbourhood_cleansed:


neighbourhood_cleansed
Westminster               10825
Tower Hamlets              6723
Kensington and Chelsea     6271
Camden                     6178
Hackney                    5740
Southwark                  5126
Lambeth                    4968
Islington                  4707
Wandsworth                 4611
Hammersmith and Fulham     4065
Brent                      2905
Lewisham                   2534
Haringey                   2443
Newham                     2433
Ealing                     2332
Barnet                     2329
Greenwich                  1898
Waltham Forest             1852
Croydon                    1691
Merton                     1594
Hounslow                   1361
Richmond upon Thames       1241
Redbridge                  1003
Hillingdon                  972
Bromley                     909
Enfield                     907
Kingston upon Thames        710
Harrow                      696
Barking and Dagenham        678
City of London              640
Bexley           

Rating is missing,False,True
Has zero reviews,,
False,71260,0
True,0,20694


In [46]:
quote_check = pd.to_numeric(
    df["price_quote_price_per_night"],
    errors="coerce"
)

both_present = price_check.notna() & quote_check.notna()

print("Listings with both price values:", both_present.sum())
print(
    "Values matching within 0.01:",
    (
        (price_check[both_present] - quote_check[both_present])
        .abs() < 0.01
    ).sum()
)

print("\nPrice-quote columns:")
print([column for column in df.columns if column.startswith("price_quote")])

Listings with both price values: 59761
Values matching within 0.01: 59761

Price-quote columns:
['price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night', 'price_quote_raw']


## Noticable information obtained form the dataset and possible Cleaing Decisons.

## Dataset Structure
Dataset contains 91,954 listings and 90 columns. The 'last_scraped' dates range from 18 september to 23 September 2026

## Missing Values 
- The target cariable, 'price', has 32,049 missing values, which is about (34.85%). 59,905 listings have an observed price.
- 15 columns are completely empty.
- The numeric 'bathrooms' column is empty but 'bathrooms_text' is missing for only 108 listings.
- 'bedrooms' has 45 missing values and 'beds' has 1,990.
- 'review_scores_rating' has 20,694 missing values. These records have zero reviews, this leads one to assume that the missing ratings represent unreviewed listings.

## Duplicate records
There are zero duplicate rows and zero repeated listing IDs. The inspection therefore provides no reason to remove duplicate records

## Price Format and unusual values
- Prices are stored as text containing a currency symbol and sometimes thousands separators. All non-missing prices could be converted to numbers during inspection without conversion failures.

## For the Observed prices
-Min:10
-Median:175
-Maximum: 75,000
-Zero or negative values:0

The highest values would require further investigation before deciding whether they are valid lsitings or errors.

## Property Characteristics
- 3,264 lsitings with zero bedrooms
- zero bathrooms
Note: The zero bedrooms should not be treated as errors because they may represent studios.
Bathroom descriptions contain information such as counts, shared facilities and half baths. This information could  support creating bathroom features despite the empty numeric bathroom column

## Potential Target Leakage
The 'price_quote_price_per_night' field has 59,761 populated values. All match the corresponding 'price' within 0.01.
This field would go on to reveal the target if used as a predictor.



## Proposed cleaning decisions

| Finding | Proposed action | Reason |
|---|---|---|
| Prices contain currency symbols and commas | Convert price text to numeric values | A regression target must be numeric |
| Some target prices are missing | Exclude these records from supervised model training | Their target values are unknown and should not be invented |
| Fifteen columns are entirely empty | Exclude empty columns from the predictor set | They contain no observed information |
| Bathroom counts are available in text | Derive bathroom counts and inspect sharing descriptions | Recover useful information from an available field |
| Some bed and bedroom counts are missing | Reassess missingness in the modelling subset, then consider training-fold imputation | Avoid unnecessary row removal and data leakage |
| Missing ratings occur on unreviewed listings | Represent review status separately and consider how to handle missing ratings | No rating is different from a poor rating |
| No duplicate records were found | Do not remove duplicates | There is no evidence supporting removal |
| Some listings have zero bedrooms | Retain initially | Zero bedrooms can describe a studio |
| Some prices are unusually high | Investigate before deciding on treatment | Extreme values are not necessarily errors |
| A price-quote field repeats the target | Exclude price-quote fields from the initial predictors | Prevent target leakage |